# Knowing When to Defer: An Ordinal Evidential Involution–Attention Network for Calibrated Knee Osteoarthritis Grading

## 00 — Core


| | Question | Paper |
|---|---|---|
| 00 core | — | Methods |
| 01 data | What is in the cohort? | Materials |
| 02 grading | Can it grade? | 3.1 |
| 03 knowing | Does it know when it's unsure? | 3.2 |
| 04 deferral | **Can it decide which films to defer?** | **3.3** |
| 05 ablation | What causes that? | 3.4 |
| 06 deployment | Can it run where radiographs are read? | 3.5 |



### Where things will live

In [ ]:
# --- Colab bootstrap: run this first in every notebook ---------------------
import os, sys, glob, shutil

IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = True        # True = results survive a disconnect. False = faster, but temporary.

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/EvIA_Knee"
elif IN_COLAB:
    PROJECT = "/content/EvIA_Knee"
else:
    PROJECT = os.path.abspath(".")

os.makedirs(PROJECT, exist_ok=True)
sys.path.insert(0, PROJECT)          # so `import evia_knee` finds the module written by 00
CACHE = os.path.join(PROJECT, "cache")
print("project:", PROJECT)

Mounted at /content/drive
project: /content/drive/MyDrive/EvIA_Knee


## Write the module

Each cell below appends one section to `evia_knee.py`.

### Imports and configuration

One `Config` dataclass holds every knob. Ablations just copy it with one field changed.

In [ ]:
%%writefile evia_knee.py
"""
evia_knee.py — shared core for the EvIA-KL study (knee osteoarthritis grading).

Companion module for:
  01_data.ipynb        cohort description and statistics
  02_grading.ipynb     grading performance and pretrained baselines
  03_knowing.ipynb     calibration and error ranking
  04_deferral.ipynb    deferral operating points (the main result)
  05_ablation.ipynb    seven one-dimensional ablation axes
  06_deployment.ipynb  efficiency and limits

Design notes that matter scientifically
---------------------------------------
* Mixed precision is OFF. `mixed_float16` without a LossScaleOptimizer produces NaN
  divergence and fp16 underflow in exactly the strength/variance quantities this study
  reports. Do not enable it.
* ECE here uses the STANDARD single bin weight, sum_m (|B_m|/N)*|acc-conf|. A common
  implementation squares the weight and silently understates calibration error.
* Grades are ORDINAL (Kellgren-Lawrence 0-4). Quadratic weighted kappa is primary;
  plain accuracy ignores that a KL0-vs-KL4 error is worse than KL2-vs-KL3.
* These are radiographs: there is no circular field to crop, vertical flips are
  anatomically invalid, and the native resolution is ~224 px.
* Images are held as uint8 and cast to float32 per batch: a quarter of the memory, and
  identical results.
"""

from __future__ import annotations

import os, re, glob, json, math, time, random, warnings
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Tuple, Optional, Callable

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import scipy.ndimage as ndi
from scipy import stats

warnings.filterwarnings("ignore")
keras.mixed_precision.set_global_policy("float32")
for _g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_g, True)

GRADE_NAMES = ["KL0 Healthy", "KL1 Doubtful", "KL2 Minimal",
               "KL3 Moderate", "KL4 Severe"]
N_GRADES = 5
OA_THRESHOLD = 2                 # KL >= 2 is definite radiographic OA: the decision that
                                 # changes management, and where readers disagree most.
OA_NAME = "definite OA (KL>=2)"

# =============================================================================
# Configuration
# =============================================================================

@dataclass
class Config:
    # --- data ---
    img_size: int = 224           # native size of the radiographs; upsampling adds nothing
    preprocess: str = "clahe"     # "none" | "clahe" | "hist_eq"
    trim_border: bool = True
    batch_size: int = 32

    # --- architecture ---
    widths: Tuple[int, ...] = (256, 320)   # EvIA branch: 14x14 then 7x7 (scratch: (64,128,256,320))
    K: int = 3                    # involution kernel size
    groups: int = 16
    reduction: int = 4
    global_branch: str = "window" # "window" (EvIA-W) | "se" (EvIA-C) | "none"
    window: int = 7
    heads: int = 4
    gate: str = "vacuity"         # "vacuity" | "mean" | "learned"
    operator: str = "evi"         # "evi" | "involution" | "conv"
    dropout: float = 0.2
    pretrained_stem: bool = False # deprecated; use `backbone`
    backbone: str = "densenet121"  # "densenet121" | "effb0" (ImageNet, full network) | "scratch"
    mlp_ratio: int = 2            # channel-mixing FFN per EvIA block (0 = off)

    # --- head and objective ---
    head: str = "ordinal"         # "ordinal" | "edl" | "softmax"
    lam_cal: float = 0.1
    lam_kl: float = 0.1
    lam_mono: float = 1.0
    kl_warmup: int = 10

    # --- optimisation ---
    epochs: int = 60              # cosine: every run trains exactly this long
    patience: int = 20            # epochs without a new best val QWK before stopping
    min_epochs: int = 40          # plateau schedule only: never stop before this
    lr_schedule: str = "cosine"   # "cosine": fixed decay to lr_min over all epochs, no early stop
                                  # "plateau": halve on val-QWK stalls, stop on patience (old rule)
    lr_min: float = 1e-6
    lr: float = 1e-3
    lr_pretrained_mult: float = 0.1  # ImageNet weights train at lr x this, in EVERY model
    seeds: Tuple[int, ...] = (42, 43, 44, 45, 46)
    class_weighted: bool = True
    class_weight_power: float = 0.5  # weights (N/(C n_c))^power; 0.5 = square-root, softer
    freeze_epochs: int = 2        # linear-probe warm-up: pretrained weights frozen this long
    tta: bool = True              # test-time left-right flip (left/right knees are mirrors)
    recalibrate: bool = True      # per-threshold Platt scaling fitted on the validation split

    # --- evaluation ---
    n_bins: int = 15
    out_dir: str = "./knee_results"

    def copy(self, **kw) -> "Config":
        d = dict(self.__dict__); d.update(kw); return Config(**d)


def set_seed(s: int):
    random.seed(s); np.random.seed(s)
    tf.random.set_seed(s); keras.utils.set_random_seed(s)

Writing evia_knee.py


### Data

Load, trim black margins, apply CLAHE, resize to 224 px. Images stay **uint8** and are cast per batch — float32 would cost four times the RAM.

In [ ]:
%%writefile -a evia_knee.py
# =============================================================================
# Data
# =============================================================================

def trim_border(img: np.ndarray, thr: int = 5) -> np.ndarray:
    """Drop uniform black margins some radiographs carry. A mild tidy-up only: these
    images already come cropped to the joint."""
    g = img.mean(-1)
    rows = np.where(g.mean(1) > thr)[0]
    cols = np.where(g.mean(0) > thr)[0]
    if rows.size == 0 or cols.size == 0:
        return img
    return img[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]


def clahe(img: np.ndarray, clip: float = 2.0, grid: int = 8) -> np.ndarray:
    """CLAHE on the luminance channel. Standard for radiographs: it lifts trabecular and
    joint-space detail without altering the global greyscale relationship."""
    try:
        import cv2
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        c = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid))
        lab[..., 0] = c.apply(lab[..., 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    except Exception:
        return img


def hist_eq(img: np.ndarray) -> np.ndarray:
    """Global histogram equalisation: the classical, non-adaptive comparator for CLAHE.
    One transfer function for the whole film, so it can wash out local joint-space detail."""
    try:
        import cv2
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        lab[..., 0] = cv2.equalizeHist(lab[..., 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    except Exception:
        return img


def load_one(path: str, cfg: Config) -> np.ndarray:
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False).numpy()
    if cfg.trim_border:
        img = trim_border(img)
    if cfg.preprocess == "clahe":
        img = clahe(img)
    elif cfg.preprocess == "hist_eq":
        img = hist_eq(img)
    img = tf.image.resize(img, (cfg.img_size, cfg.img_size), method="bilinear").numpy()
    return np.clip(img, 0, 255).astype("uint8")      # uint8: 4x less RAM than float32


def build_array(paths: List[str], labels: np.ndarray, cfg: Config,
                verbose: bool = True) -> Tuple[np.ndarray, np.ndarray]:
    X = np.zeros((len(paths), cfg.img_size, cfg.img_size, 3), "uint8")
    for i, p in enumerate(paths):
        X[i] = load_one(p, cfg)
        if verbose and i % 250 == 0:
            print(f"    {i}/{len(paths)}")
    return X, labels.astype("int32")


# Explicit paths always win. If auto-discovery fails, set one line and move on:
#     E.DATA_ROOTS["knee"] = "/content/knee"
DATA_ROOTS: Dict[str, str] = {}

_SEARCH = ["/kaggle/input/{p}", "/kaggle/input/*/{p}",      # Kaggle notebooks
           "/content/{p}", "/content/*/{p}",                # Colab
           "/content/drive/MyDrive/{p}",                    # Colab + mounted Drive
           "./{p}", "../{p}", "./data/{p}"]                 # local


def cached_split(paths, labels, cfg: Config, tag: str, cache_dir: str = None,
                 verbose: bool = False):
    """build_array(), but saved to disk so a restarted session skips the slow decode.

    The cache key includes image size and preprocessing, so changing either rebuilds.
    Each split is ~1 GB at 384 px -- far smaller than re-downloading the raw data.
    """
    if not cache_dir:
        return build_array(paths, labels, cfg, verbose)
    os.makedirs(cache_dir, exist_ok=True)
    key = f"{tag}_{cfg.img_size}_{cfg.preprocess}_{int(cfg.trim_border)}_{len(paths)}"
    fx = os.path.join(cache_dir, key + "_x.npy")
    fy = os.path.join(cache_dir, key + "_y.npy")
    if os.path.exists(fx) and os.path.exists(fy):
        print(f"  {tag}: loaded from cache")
        return np.load(fx), np.load(fy)
    X, Y = build_array(paths, labels, cfg, verbose)
    np.save(fx, X); np.save(fy, Y)
    print(f"  {tag}: built and cached ({X.nbytes/1e9:.2f} GB)")
    return X, Y


def find_dataset(name: str) -> str:
    """Locate a dataset folder. Checks DATA_ROOTS first, then common mount points."""
    if name in DATA_ROOTS:
        r = DATA_ROOTS[name]
        if os.path.isdir(r):
            return r
        raise FileNotFoundError(f"DATA_ROOTS['{name}'] = {r} is not a directory")

    pats = {"knee": ["*knee*", "*Knee*", "*KneeOA*", "*osteoarthritis*"]}[name]
    for tmpl in _SEARCH:
        for p in pats:
            for c in sorted(glob.glob(tmpl.format(p=p))):
                if os.path.isdir(c):
                    return c
    raise FileNotFoundError(
        f"{name} not found. Searched {_SEARCH} .\n"
        f"Fix with one line:  E.DATA_ROOTS['{name}'] = '/path/to/folder'\n"
        f"Or download it:     E.download_kaggle('{name}')")


def download_kaggle(name: str, dest: str = None) -> str:
    """Fetch a dataset from Kaggle and register its path.

    Needs ~/.kaggle/kaggle.json (Kaggle -> Settings -> API -> Create New Token).
    A plain Kaggle dataset: no competition rules to accept.
    """
    import subprocess, zipfile
    if name != "knee":
        raise ValueError(f"no download recipe for '{name}'")
    dest = dest or f"/content/{name}"
    os.makedirs(dest, exist_ok=True)

    # Mirrors come and go, so try each candidate until one works.
    last = None
    for kind, slug in KNEE_MIRRORS:
        cmd = (["kaggle", "competitions", "download", "-c", slug, "-p", dest]
               if kind == "competition" else
               ["kaggle", "datasets", "download", "-d", slug, "-p", dest])
        print(" ".join(cmd))
        r = subprocess.run(cmd)
        if r.returncode == 0 and glob.glob(os.path.join(dest, "*.zip")):
            break
        last = slug
        print(f"  '{slug}' failed; trying the next mirror")
    else:
        raise RuntimeError(
            f"every mirror for '{name}' failed (last: {last}). Search Kaggle for the "
            f"dataset, download it manually, then set E.DATA_ROOTS['{name}'].")

    for z in glob.glob(os.path.join(dest, "*.zip")):
        print("unzipping", os.path.basename(z))
        with zipfile.ZipFile(z) as f:
            f.extractall(dest)
        os.remove(z)

    DATA_ROOTS[name] = dest
    print("registered DATA_ROOTS[%r] = %s" % (name, dest))
    return dest


# Folder names seen across mirrors of this dataset, mapped to KL grade.
_CLASS_ALIASES = {
    "0": 0, "normal": 0, "healthy": 0, "grade0": 0, "kl0": 0,
    "1": 1, "doubtful": 1, "grade1": 1, "kl1": 1,
    "2": 2, "mild": 2, "minimal": 2, "grade2": 2, "kl2": 2,
    "3": 3, "moderate": 3, "grade3": 3, "kl3": 3,
    "4": 4, "severe": 4, "grade4": 4, "kl4": 4,
}

# Kaggle mirrors of the Mendeley "Knee Osteoarthritis Severity Grading Dataset".
# Tried in order. If you already have a copy: DATA_ROOTS["knee"] = "/path".
KNEE_MIRRORS = [("dataset", "shashwatwork/knee-osteoarthritis-dataset-with-severity"),
                ("dataset", "tommyngx/kneeoa")]


def load_knee(root: Optional[str] = None, cfg: Config = None, seed: int = 42,
              split_frac=(0.70, 0.10, 0.20)):
    """Knee radiographs, one folder per KL grade.

    Uses the dataset's own train/val/test folders when present so numbers stay
    comparable with other papers; otherwise makes a stratified 70/10/20 split.
    """
    from sklearn.model_selection import train_test_split
    root = root or find_dataset("knee")

    files = []
    for e in ("*.png", "*.jpg", "*.jpeg", "*.bmp"):
        files += glob.glob(os.path.join(root, "**", e), recursive=True)
    if not files:
        raise FileNotFoundError(f"no images under {root}")

    paths, ys, shipped = [], [], []
    for f in files:
        parts = [p.lower().replace(" ", "").replace("_", "") for p in f.split(os.sep)]
        grade = next((_CLASS_ALIASES[p] for p in reversed(parts) if p in _CLASS_ALIASES), None)
        if grade is None:
            continue                                   # not inside a class folder
        sp = next((p for p in parts
                   if p in ("train", "val", "valid", "validation", "test")), None)
        paths.append(f); ys.append(grade)
        shipped.append({"valid": "val", "validation": "val"}.get(sp, sp))

    if not paths:
        raise FileNotFoundError(
            f"{len(files)} images under {root} but no recognised class folders.\n"
            f"Expected folders named 0-4 or Normal/Doubtful/Mild/Moderate/Severe.")

    paths = np.array(paths); y = np.array(ys, "int32"); shipped = np.array(shipped)

    if set(x for x in shipped if x) >= {"train", "test"}:
        idx = {k: np.where(shipped == k)[0] for k in ("train", "val", "test")}
        if len(idx["val"]) == 0:                       # train/test only: carve a val set
            tr, va = train_test_split(idx["train"], test_size=.125,
                                      stratify=y[idx["train"]], random_state=seed)
            idx = dict(train=tr, val=va, test=idx["test"])
        print("  using the dataset's own train/val/test split")
    else:
        i = np.arange(len(y))
        tv, te = train_test_split(i, test_size=split_frac[2], stratify=y, random_state=seed)
        tr, va = train_test_split(tv, test_size=split_frac[1] / (1 - split_frac[2]),
                                  stratify=y[tv], random_state=seed)
        idx = dict(train=tr, val=va, test=te)
        print("  no shipped split found; made a stratified 70/10/20 split")

    return dict(name="Knee OA (KL 0-4)", paths=paths, y=y, idx=idx)


def class_weights(y: np.ndarray, C: int = N_GRADES, power: float = 1.0) -> np.ndarray:
    """Inverse-frequency weights raised to `power`. Full weights (power 1) gave KL4
    ~6.7x the weight of KL0 and pushed KL3 knees into KL4; the square root keeps the
    rare grades visible without overshooting."""
    cnt = np.bincount(y, minlength=C).astype("float64")
    return ((len(y) / (C * np.maximum(cnt, 1))) ** power).astype("float32")


def to_float(x):
    """uint8 image batch -> float32 in [0,1]. Applied per batch, never to the whole set."""
    return tf.cast(x, tf.float32) / 255.0


def augment(x: tf.Tensor) -> tf.Tensor:
    """Left-right flip is valid: left and right knees are mirror images.

    A VERTICAL flip is not. A radiograph has a fixed superior-inferior orientation, so an
    upside-down knee is not a knee, and the KL criteria are read with that orientation
    assumed. Rotation is also left out: these films arrive cropped and registered to a
    standard AP projection, so rotating them simulates a positioning error the cohort
    does not contain. What remains is exposure variation, which is real between machines."""
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_brightness(x, 0.08)            # exposure variation
    x = tf.image.random_contrast(x, 0.9, 1.1)
    return tf.clip_by_value(x, 0.0, 1.0)

Appending to evia_knee.py


### Operators

`EvI` is the evidential involution: it returns the feature map plus a vacuity map, free. `EvIABlock` fuses it with a global branch using a gate with no parameters.

In [ ]:
%%writefile -a evia_knee.py
# =============================================================================
# Operators
# =============================================================================

class EvI(layers.Layer):
    """Evidential involution (Alzahrani 2026, Mathematics 14:3310).

        z = span(relu(bn(reduce(h))))     e = softplus(z)     alpha = e + 1
        Sbar = mean_g sum_k alpha_gk      wbar = alpha / S     u = K^2 / Sbar

    Returns (output, vacuity_map, strength_map).
    """
    def __init__(self, channels, k=3, groups=16, reduction=4, **kw):
        super().__init__(**kw)
        self.C, self.K, self.G0, self.r = channels, k, groups, reduction

    def build(self, s):
        C = self.C
        self.G = math.gcd(self.G0, C)
        self.reduce = layers.Conv2D(max(C // self.r, 8), 1, use_bias=False)
        self.bn = layers.BatchNormalization()
        self.act = layers.Activation("relu")
        self.span = layers.Conv2D(self.K * self.K * self.G, 1)
        super().build(s)

    def call(self, x, training=None):
        B, H, W = tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2]
        C, K, G = self.C, self.K, self.G
        z = self.span(self.act(self.bn(self.reduce(x), training=training)))
        alpha = tf.reshape(tf.nn.softplus(z) + 1.0, [B, H, W, G, K * K])
        S = tf.reduce_sum(alpha, -1, keepdims=True)
        w = alpha / S
        p = tf.image.extract_patches(x, [1, K, K, 1], [1, 1, 1, 1], [1, 1, 1, 1], "SAME")
        p = tf.reshape(p, [B, H, W, K * K, G, C // G])
        p = tf.transpose(p, [0, 1, 2, 4, 3, 5])
        out = tf.reshape(tf.reduce_sum(w[..., None] * p, -2), [B, H, W, C])
        Sbar = tf.reduce_mean(S[..., 0], -1)
        u = tf.cast(K * K, tf.float32) / Sbar
        return out, u, Sbar


class Involution(layers.Layer):
    """Standard unnormalised involution (Li et al. 2021). Ablation control."""
    def __init__(self, channels, k=3, groups=16, reduction=4, **kw):
        super().__init__(**kw)
        self.C, self.K, self.G0, self.r = channels, k, groups, reduction

    def build(self, s):
        C = self.C
        self.G = math.gcd(self.G0, C)
        self.reduce = layers.Conv2D(max(C // self.r, 8), 1, use_bias=False)
        self.bn = layers.BatchNormalization(); self.act = layers.Activation("relu")
        self.span = layers.Conv2D(self.K * self.K * self.G, 1)
        super().build(s)

    def call(self, x, training=None):
        B, H, W = tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2]
        C, K, G = self.C, self.K, self.G
        kern = tf.reshape(self.span(self.act(self.bn(self.reduce(x), training=training))),
                          [B, H, W, G, K * K])
        p = tf.image.extract_patches(x, [1, K, K, 1], [1, 1, 1, 1], [1, 1, 1, 1], "SAME")
        p = tf.reshape(p, [B, H, W, K * K, G, C // G])
        p = tf.transpose(p, [0, 1, 2, 4, 3, 5])
        out = tf.reshape(tf.reduce_sum(kern[..., None] * p, -2), [B, H, W, C])
        ones = tf.ones(tf.shape(out)[:3])
        return out, ones * 0.5, ones * float(K * K * 2)


class SEBranch(layers.Layer):
    def __init__(self, c, r=4, **kw):
        super().__init__(**kw)
        self.gap = layers.GlobalAveragePooling2D(keepdims=True)
        self.d1 = layers.Dense(max(c // r, 8), activation="relu")
        self.d2 = layers.Dense(c, activation="sigmoid")
    def call(self, x):
        return x * self.d2(self.d1(self.gap(x)))


class WindowBranch(layers.Layer):
    """Non-overlapping windowed multi-head self-attention."""
    def __init__(self, c, heads=4, win=6, **kw):
        super().__init__(**kw)
        self.c, self.win = c, win
        self.ln = layers.LayerNormalization()
        self.mha = layers.MultiHeadAttention(num_heads=heads, key_dim=max(c // heads, 8))

    def call(self, x):
        B, H, W, C = tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2], self.c
        win = self.win
        pad = (win - H % win) % win
        x = tf.pad(x, [[0, 0], [0, pad], [0, pad], [0, 0]])
        Hp = H + pad
        nh = Hp // win
        y = tf.reshape(x, [B, nh, win, nh, win, C])
        y = tf.transpose(y, [0, 1, 3, 2, 4, 5])
        y = tf.reshape(y, [B * nh * nh, win * win, C])
        y = self.mha(self.ln(y), self.ln(y))
        y = tf.reshape(y, [B, nh, nh, win, win, C])
        y = tf.transpose(y, [0, 1, 3, 2, 4, 5])
        y = tf.reshape(y, [B, Hp, Hp, C])
        return y[:, :H, :W, :]


class EvIABlock(layers.Layer):
    """MetaFormer-style block (Yu et al., CVPR 2022): a token mixer, then a channel mixer.

    Token mixer = local operator + global branch, combined by the vacuity-gated fusion.
    Channel mixer = a 1x1 expand/contract FFN. The original EvIA block had no channel
    mixing at all: involution and attention both output convex combinations of their
    inputs, so every learnable cross-channel transform sat in the 1x1 projection alone.
    """
    def __init__(self, c, cfg: Config, **kw):
        super().__init__(**kw)
        self.cfg = cfg
        self.proj = layers.Conv2D(c, 1, use_bias=False)
        self.bn0 = layers.BatchNormalization(); self.a0 = layers.Activation("relu")

        if cfg.operator == "evi":
            self.op = EvI(c, cfg.K, cfg.groups, cfg.reduction)
        elif cfg.operator == "involution":
            self.op = Involution(c, cfg.K, cfg.groups, cfg.reduction)
        else:
            self.op = None
            self.conv = layers.DepthwiseConv2D(cfg.K, padding="same", use_bias=False)

        if cfg.global_branch == "window":
            self.glob = WindowBranch(c, cfg.heads, cfg.window)
        elif cfg.global_branch == "se":
            self.glob = SEBranch(c, cfg.reduction)
        else:
            self.glob = None

        if cfg.gate == "learned":
            # Keras 3: shape is the first argument, name is keyword-only.
            self.gate_w = self.add_weight(shape=(), initializer="zeros",
                                          trainable=True, name="gate_w")
        self.bn1 = layers.BatchNormalization(); self.a1 = layers.Activation("relu")
        if cfg.mlp_ratio > 0:
            self.bn2 = layers.BatchNormalization()
            self.ffn1 = layers.Conv2D(c * cfg.mlp_ratio, 1, activation="gelu")
            self.ffn2 = layers.Conv2D(c, 1)
        self.drop = layers.Dropout(cfg.dropout)

    def call(self, x, training=None):
        cfg = self.cfg
        h = self.a0(self.bn0(self.proj(x), training=training))

        if self.op is not None:
            y_loc, u, Sbar = self.op(h, training=training)
        else:
            y_loc = self.conv(h)
            ones = tf.ones(tf.shape(y_loc)[:3])
            u, Sbar = ones * 0.5, ones * float(cfg.K ** 2 * 2)

        if self.glob is None:
            z = y_loc
        else:
            y_glob = self.glob(h)
            if cfg.gate == "vacuity":
                g = (1.0 / (1.0 + u))[..., None]          # = Sbar/(Sbar+K^2)
            elif cfg.gate == "learned":
                g = tf.sigmoid(self.gate_w)
            else:
                g = 0.5
            z = g * y_loc + (1.0 - g) * y_glob

        z = self.a1(self.bn1(z, training=training) + h)           # token mixing + residual
        if cfg.mlp_ratio > 0:                                       # channel mixing + residual
            z = z + self.ffn2(self.ffn1(self.bn2(z, training=training)))
        return self.drop(z, training=training), u, Sbar

Appending to evia_knee.py


### Ordinal head

The one new piece. Four Beta posteriors, one per grade boundary, instead of a flat Dirichlet over five classes.

In [ ]:
%%writefile -a evia_knee.py
# =============================================================================
# Ordinal evidential head
# =============================================================================

class OrdinalEvidentialHead(layers.Layer):
    """Cumulative-link evidential head for Kellgren-Lawrence grades.

    For each of the K-1 = 4 thresholds k, the head emits two non-negative evidence
    values, giving a Beta posterior over the cumulative probability Q_k = P(y > k):

        alpha_k = softplus(a_k) + 1 ,  beta_k = softplus(b_k) + 1
        S_k     = alpha_k + beta_k
        Q_k     = alpha_k / S_k
        u_k     = 2 / S_k                      (binary vacuity at threshold k)

    Grade probabilities are assembled from the cumulative chain, and the overall
    predictive vacuity is the mean threshold vacuity. The *targeted* vacuity
    u_ref = u_{t-1} is the uncertainty at the single actionable boundary, KL1 vs KL2 --
    the call that decides whether a knee has definite radiographic OA, and the call on
    which expert readers disagree most. The deferral rule uses it.
    """
    def __init__(self, n_grades=N_GRADES, **kw):
        super().__init__(**kw)
        self.T = n_grades - 1
        self.dense = layers.Dense(2 * self.T, name="ordinal_evidence")

    def call(self, feat):
        raw = self.dense(feat)
        raw = tf.reshape(raw, [-1, self.T, 2])
        ab = tf.nn.softplus(raw) + 1.0                      # (B,T,2)
        alpha, beta = ab[..., 0], ab[..., 1]
        S = alpha + beta
        Q = alpha / S                                       # P(y > k)
        u = 2.0 / S                                         # per-threshold vacuity
        return dict(alpha=alpha, beta=beta, S=S, Q=Q, u=u)


def grades_from_cumulative(Q: tf.Tensor) -> tf.Tensor:
    """Assemble p(y=k) from the cumulative chain, enforcing non-negativity.
    Q: (B,T) with Q_k = P(y>k). Returns (B,T+1)."""
    one = tf.ones_like(Q[:, :1])
    left = tf.concat([one, Q], axis=1)            # P(y > k-1), k = 0..T
    right = tf.concat([Q, tf.zeros_like(Q[:, :1])], axis=1)   # P(y > k)
    p = tf.maximum(left - right, 1e-6)
    return p / tf.reduce_sum(p, -1, keepdims=True)


def ordinal_loss(y, out, cw, klw, cfg: Config):
    """Beta Bayes-risk per threshold + monotonicity + KL-to-uniform on misleading evidence."""
    T = tf.shape(out["Q"])[1]
    ks = tf.range(T, dtype=y.dtype)
    t = tf.cast(y[:, None] > ks[None, :], tf.float32)        # (B,T) targets 1[y>k]

    Q, S = out["Q"], out["S"]
    err = (t - Q) ** 2
    var = Q * (1.0 - Q) / (S + 1.0)
    per_thr = err + var                                      # Beta Bayes risk of Brier

    w = tf.gather(cw, y)[:, None] if cfg.class_weighted else 1.0
    L = tf.reduce_mean(w * per_thr)

    mono = tf.reduce_mean(tf.nn.relu(Q[:, 1:] - Q[:, :-1]))  # Q must be non-increasing

    # KL(Beta(alpha_t,beta_t) || Beta(1,1)) on the misleading side only
    a_t = tf.where(t > 0.5, tf.ones_like(out["alpha"]), out["alpha"])
    b_t = tf.where(t > 0.5, out["beta"], tf.ones_like(out["beta"]))
    S_t = a_t + b_t
    kl = (tf.math.lgamma(S_t) - tf.math.lgamma(a_t) - tf.math.lgamma(b_t)
          - tf.math.lgamma(2.0)
          + (a_t - 1.0) * (tf.math.digamma(a_t) - tf.math.digamma(S_t))
          + (b_t - 1.0) * (tf.math.digamma(b_t) - tf.math.digamma(S_t)))
    return L, cfg.lam_mono * mono, klw * tf.reduce_mean(kl)


def flat_edl_loss(y, logits, cw, klw, C=N_GRADES, class_weighted=True):
    """Flat Dirichlet EDL head (ablation control: no ordinal structure)."""
    y1 = tf.one_hot(y, C)
    alpha = tf.nn.softplus(logits) + 1.0
    S = tf.reduce_sum(alpha, -1, keepdims=True)
    p = alpha / S
    err = tf.reduce_sum((y1 - p) ** 2, -1)
    var = tf.reduce_sum(p * (1 - p) / (S + 1), -1)
    w = tf.reduce_sum(y1 * cw, -1) if class_weighted else 1.0
    at = y1 + (1 - y1) * alpha
    St = tf.reduce_sum(at, -1, keepdims=True)
    kl = (tf.math.lgamma(St[..., 0]) - tf.math.lgamma(tf.cast(C, tf.float32))
          - tf.reduce_sum(tf.math.lgamma(at), -1)
          + tf.reduce_sum((at - 1) * (tf.math.digamma(at) - tf.math.digamma(St)), -1))
    return tf.reduce_mean(w * (err + var)) + klw * tf.reduce_mean(kl)


def softmax_loss(y, logits, cw, C=N_GRADES, class_weighted=True):
    y1 = tf.one_hot(y, C)
    ce = tf.nn.softmax_cross_entropy_with_logits(y1, logits)
    w = tf.reduce_sum(y1 * cw, -1) if class_weighted else 1.0
    return tf.reduce_mean(ce * w)


def soft_calibration(p, y, n_bins=15):
    """Differentiable soft-binned calibration surrogate (Barfoot et al. 2024)."""
    conf = tf.reduce_max(p, -1)
    acc = tf.cast(tf.equal(tf.argmax(p, -1, output_type=y.dtype), y), tf.float32)
    centres = tf.linspace(0.0, 1.0, n_bins)
    width = 1.0 / tf.cast(n_bins, tf.float32)
    d = tf.abs(conf[:, None] - centres[None, :])
    m = tf.nn.relu(1.0 - d / width)                       # triangular soft assignment
    mass = tf.reduce_sum(m, 0) + 1e-6
    bin_conf = tf.reduce_sum(m * conf[:, None], 0) / mass
    bin_acc = tf.reduce_sum(m * acc[:, None], 0) / mass
    wgt = mass / tf.reduce_sum(mass)
    return tf.reduce_sum(wgt * tf.abs(bin_acc - bin_conf))

Appending to evia_knee.py


### Model and training

Assembly, loss selection and the training loop with early stopping.

In [ ]:
%%writefile -a evia_knee.py
# =============================================================================
# Model
# =============================================================================

# name: (keras.applications constructor, preprocessing, 14x14 layer, 7x7 layer) at 224 px
BACKBONES = {
    "densenet121": ("DenseNet121", "densenet", "conv4_block24_concat", "relu"),
    "effb0":       ("EfficientNetB0", "efficientnet", "block5c_add", "top_activation"),
}


def build_stem(cfg: Config):
    """Feature extractor for EvIA-KL.

    Pretrained ("densenet121", "effb0"): the WHOLE ImageNet network, returning two maps:
      mid -- the 14x14 map at the end of the penultimate stage, fed to the EvIA branch
             (where its vacuity maps are read, at the scale of the joint);
      top -- the final 7x7 map, the same features the matching baseline classifies.
    EvIA-KL pools both and concatenates them, so the EvIA branch adds to the baseline's
    representation instead of replacing it. Pixels are mapped to 0-255 and passed
    through the backbone's own preprocessing, exactly as in build_baseline().
    "scratch": the original randomly initialised two-convolution stem (one map).
    """
    S = cfg.img_size
    if cfg.backbone in BACKBONES:
        ctor, prep, mid_name, top_name = BACKBONES[cfg.backbone]
        base = getattr(keras.applications, ctor)(
            include_top=False, weights="imagenet", input_shape=(S, S, 3))
        trunk = keras.Model(base.input, [base.get_layer(mid_name).output,
                                         base.get_layer(top_name).output],
                            name=f"{cfg.backbone}_trunk")
        pre = getattr(keras.applications, prep).preprocess_input
        inp = keras.Input((S, S, 3))
        x = layers.Rescaling(255.0)(inp)
        x = layers.Lambda(pre, name="preprocess")(x)
        return keras.Model(inp, trunk(x), name="pretrained_trunk")
    if cfg.backbone != "scratch":
        raise ValueError(f"unknown backbone {cfg.backbone!r}")
    return keras.Sequential([
        layers.Conv2D(32, 3, strides=2, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.Conv2D(cfg.widths[0], 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPool2D(2),
    ], name="stem")


class EvIAKL(keras.Model):
    """EvIA-KL: evidential involution-attention network for Kellgren-Lawrence grading."""

    def __init__(self, cfg: Config, **kw):
        super().__init__(**kw)
        self.cfg = cfg
        self.stem = build_stem(cfg)
        self.blocks = [EvIABlock(w, cfg, name=f"evia_{i}") for i, w in enumerate(cfg.widths)]
        self.pools = [layers.MaxPool2D(2) for _ in cfg.widths[:-1]]
        self.gap = layers.GlobalAveragePooling2D()
        self.gap_top = layers.GlobalAveragePooling2D()     # backbone's own final features
        self.head_drop = layers.Dropout(cfg.dropout)       # as in the baselines
        if cfg.head == "ordinal":
            self.head = OrdinalEvidentialHead(N_GRADES)
        else:
            self.head = layers.Dense(N_GRADES, name="logits")

    def call(self, x, training=None):
        st = self.stem(x, training=training)
        h, top = (st[0], st[1]) if isinstance(st, (list, tuple)) else (st, None)
        vac = []
        for i, blk in enumerate(self.blocks):
            h, u, S = blk(h, training=training)
            vac.append(u)
            if i < len(self.pools):
                h = self.pools[i](h)
        f = self.gap(h)
        if top is not None:                                 # EvIA branch + backbone features
            f = tf.concat([f, self.gap_top(top)], axis=-1)
        f = self.head_drop(f, training=training)
        out = self.head(f)
        if self.cfg.head == "ordinal":
            out = dict(out)
            out["p"] = grades_from_cumulative(out["Q"])
            out["u_cls"] = tf.reduce_mean(out["u"], -1)
            out["u_ref"] = out["u"][:, OA_THRESHOLD - 1]
        else:
            logits = out
            if self.cfg.head == "edl":
                alpha = tf.nn.softplus(logits) + 1.0
                S = tf.reduce_sum(alpha, -1, keepdims=True)
                p = alpha / S
                u_cls = float(N_GRADES) / S[..., 0]
            else:
                p = tf.nn.softmax(logits, -1)
                u_cls = 1.0 - tf.reduce_max(p, -1)
            out = dict(logits=logits, p=p, u_cls=u_cls, u_ref=u_cls)
        out["vacuity_maps"] = vac
        return out

    def pretrained_variables(self):
        return list(self.stem.trainable_variables) if self.cfg.backbone != "scratch" else []


def model_loss(model: EvIAKL, x, y, cw, klw, cfg: Config, training: bool):
    out = model(x, training=training)
    p = out["p"]
    if cfg.head == "ordinal":
        base, mono, kl = ordinal_loss(y, out, cw, klw, cfg)
        L = base + mono + kl
    elif cfg.head == "edl":
        L = flat_edl_loss(y, out["logits"], cw, klw, N_GRADES, cfg.class_weighted)
    else:
        L = softmax_loss(y, out["logits"], cw, N_GRADES, cfg.class_weighted)
    if cfg.lam_cal > 0:
        L += cfg.lam_cal * soft_calibration(p, y, cfg.n_bins)
    return L, out

# =============================================================================
# Training
# =============================================================================

def make_ds(X, Y, cfg: Config, training: bool, seed: int = 42):
    ds = tf.data.Dataset.from_tensor_slices((X, Y))
    if training:
        ds = ds.shuffle(min(len(X), 2048), seed=seed, reshuffle_each_iteration=True)
        # cast to float BEFORE augmenting: augment() expects values in [0,1]
        ds = ds.map(lambda a, b: (augment(to_float(a)), b),
                    num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(lambda a, b: (to_float(a), b), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(cfg.batch_size).prefetch(tf.data.AUTOTUNE)


def pretrained_variables(model):
    """ImageNet-initialised weights of any model in this study (EvIA-KL or a baseline)."""
    if hasattr(model, "pretrained_variables"):
        return model.pretrained_variables()
    subs = [l for l in model.layers if isinstance(l, keras.Model)]   # the applications backbone
    return [v for l in subs for v in l.trainable_variables]


class TwoRateOptimizer:
    """The one optimiser rule for every model: Adam at `lr` for new layers and at
    `lr * lr_pretrained_mult` for ImageNet weights. A model with no pretrained weights
    (scratch EvIA-KL) gets a single rate. Gradients are clipped jointly (norm 5)."""

    def __init__(self, model, cfg: Config):
        self.mult = cfg.lr_pretrained_mult
        pre = pretrained_variables(model)
        ids = {id(v) for v in pre}
        self.new = [v for v in model.trainable_variables if id(v) not in ids]
        self.pre = pre
        self.vars = self.new + self.pre
        self.opt_new = keras.optimizers.Adam(cfg.lr)
        self.opt_pre = keras.optimizers.Adam(cfg.lr * self.mult) if pre else None
        # build optimiser state now: inside tf.function, variables may only be created on
        # the first trace, and the warm-up switch retraces the step once
        self.opt_new.build(self.new)
        if self.opt_pre is not None:
            self.opt_pre.build(self.pre)

    def apply(self, grads, train_pre=True):
        """train_pre=False during the warm-up: new layers learn, ImageNet weights wait
        (linear probing then fine-tuning; Kumar et al., ICLR 2022)."""
        grads, _ = tf.clip_by_global_norm(grads, 5.0)
        n = len(self.new)
        self.opt_new.apply_gradients(zip(grads[:n], self.new))
        if self.opt_pre is not None and train_pre:
            self.opt_pre.apply_gradients(zip(grads[n:], self.pre))

    def set_lr(self, lr):
        for opt, val in ((self.opt_new, lr), (self.opt_pre, lr * self.mult)):
            if opt is None:
                continue
            try:                                       # Keras 3
                opt.learning_rate.assign(val)
            except Exception:                          # older APIs
                opt.learning_rate = val


def train_model(data, cfg: Config, seed: int, verbose: int = 1):
    set_seed(seed)
    Xtr, Ytr = data["train"]; Xva, Yva = data["val"]
    cw = tf.constant(class_weights(Ytr, power=cfg.class_weight_power))
    model = EvIAKL(cfg)
    _ = model(tf.zeros([1, cfg.img_size, cfg.img_size, 3]), training=False)   # build variables
    opt = TwoRateOptimizer(model, cfg)

    tr = make_ds(Xtr, Ytr, cfg, True, seed)
    va = make_ds(Xva, Yva, cfg, False)

    @tf.function
    def step(xb, yb, klw, train_pre=True):
        with tf.GradientTape() as tape:
            L, _ = model_loss(model, xb, yb, cw, klw, cfg, True)
        opt.apply(tape.gradient(L, opt.vars), train_pre)
        return L

    @tf.function
    def vstep(xb, yb, klw):
        L, _ = model_loss(model, xb, yb, cw, klw, cfg, False)
        return L

    def val_qwk():
        """QWK on the validation split -- the checkpoint criterion.

        Validation LOSS cannot pick the checkpoint here: it contains the KL term, whose
        weight ramps from 0 to lam_kl over the first kl_warmup epochs, so the loss rises
        during warm-up even while grading improves. Selecting on it restored near-initial
        weights. QWK is the paper's primary metric and does not move with the KL schedule.
        """
        preds = []
        for xb, _ in va:
            preds.append(decode(model(xb, training=False)["p"].numpy()))
        return quadratic_weighted_kappa(Yva, np.concatenate(preds))

    best, bw, wait, hist = -np.inf, None, 0, []
    lr_now = cfg.lr
    for ep in range(cfg.epochs):
        klw = tf.constant(cfg.lam_kl * min(1.0, (ep + 1) / max(cfg.kl_warmup, 1)), tf.float32)
        for xb, yb in tr:
            step(xb, yb, klw, ep >= cfg.freeze_epochs)
        vl = float(np.mean([vstep(xb, yb, klw).numpy() for xb, yb in va]))  # logged only
        vq = val_qwk()
        hist.append(dict(epoch=ep, val_loss=vl, val_qwk=vq))
        if vq > best + 1e-4:
            best, bw, wait = vq, model.get_weights(), 0
        else:
            wait += 1
            if cfg.lr_schedule == "plateau":
                if wait % (cfg.patience // 2) == 0:
                    lr_now = max(lr_now * 0.5, cfg.lr_min)
                    opt.set_lr(lr_now)
                if wait >= cfg.patience and ep + 1 >= cfg.min_epochs:
                    break
        if cfg.lr_schedule == "cosine":                 # same fixed schedule for every model
            lr_now = cosine_lr(cfg, ep + 1)
            opt.set_lr(lr_now)
        if verbose and ep % 5 == 0:
            print(f"    ep {ep:3d}  val_qwk {vq:.3f}  val_loss {vl:.4f}  lr {lr_now:.1e}")
    if verbose:
        print(f"    best val QWK {best:.3f}")
    if bw is not None:
        model.set_weights(bw)
    return model, hist


def predict(model, X, cfg: Config, bs: int = None):
    bs = bs or cfg.batch_size
    P, U, UR = [], [], []
    for i in range(0, len(X), bs):
        xb = to_float(tf.constant(X[i:i + bs]))
        o = model(xb, training=False)
        p, u, ur = np.asarray(o["p"]), np.asarray(o["u_cls"]), np.asarray(o["u_ref"])
        if cfg.tta:                                    # mirror image of the same knee
            o2 = model(tf.image.flip_left_right(xb), training=False)
            p = (p + np.asarray(o2["p"])) / 2
            u = (u + np.asarray(o2["u_cls"])) / 2
            ur = (ur + np.asarray(o2["u_ref"])) / 2
        P.append(p); U.append(u); UR.append(ur)
    return np.concatenate(P), np.concatenate(U), np.concatenate(UR)

Appending to evia_knee.py


### Metrics

QWK (primary), accuracy, calibration errors, AURC.

In [ ]:
%%writefile -a evia_knee.py
# =============================================================================
# Metrics
# =============================================================================

def quadratic_weighted_kappa(y_true, y_pred, C: int = N_GRADES) -> float:
    """The standard DR grading metric: penalises errors by the SQUARE of grade distance."""
    O = np.zeros((C, C))
    for a, b in zip(y_true, y_pred):
        O[a, b] += 1
    w = np.array([[((i - j) ** 2) / ((C - 1) ** 2) for j in range(C)] for i in range(C)])
    ha = np.bincount(y_true, minlength=C).astype("float64")
    hb = np.bincount(y_pred, minlength=C).astype("float64")
    E = np.outer(ha, hb); E = E * O.sum() / max(E.sum(), 1e-12)
    den = (w * E).sum()
    return float(1.0 - (w * O).sum() / den) if den > 0 else 0.0


def ece(p, y, n_bins: int = 15) -> float:
    """Standard ECE:  sum_m (|B_m|/N) * |acc_m - conf_m|.
    NOTE: weighting by (|B_m|/N)^2 is a common bug that understates the error."""
    conf = p.max(1); acc = (p.argmax(1) == y).astype("float64")
    edges = np.linspace(0, 1, n_bins + 1); e = 0.0
    for i in range(n_bins):
        m = (conf > edges[i]) & (conf <= edges[i + 1])
        if m.sum():
            e += (m.sum() / len(p)) * abs(acc[m].mean() - conf[m].mean())
    return float(e)


def ace(p, y, n_bins: int = 15) -> float:
    conf = p.max(1); acc = (p.argmax(1) == y).astype("float64")
    order = np.argsort(conf); gaps = []
    for idx in np.array_split(order, n_bins):
        if idx.size:
            gaps.append(abs(acc[idx].mean() - conf[idx].mean()))
    return float(np.mean(gaps))


def brier(p, y, C=N_GRADES) -> float:
    oh = np.eye(C)[y]
    return float(((p - oh) ** 2).sum(1).mean())


def nll(p, y) -> float:
    return float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)).mean())


def aurc(unc, correct) -> float:
    """Area under the risk-coverage curve; `unc` ranks predictions least to most certain."""
    o = np.argsort(unc)
    err = (~correct.astype(bool))[o].astype(float)
    risk = np.cumsum(err) / np.arange(1, len(err) + 1)
    return float(risk.mean())


def oa_metrics(p, y, thr=OA_THRESHOLD) -> dict:
    """Binary endpoint: definite radiographic OA, KL >= 2."""
    from sklearn.metrics import roc_auc_score, roc_curve
    yb = (y >= thr).astype(int)
    sb = p[:, thr:].sum(1)
    if yb.sum() == 0 or yb.sum() == len(yb):
        return dict(oa_auc=np.nan, sens_at_90spec=np.nan, spec_at_90sens=np.nan)
    auc = float(roc_auc_score(yb, sb))
    fpr, tpr, _ = roc_curve(yb, sb)
    spec = 1 - fpr
    s90 = float(tpr[np.argmin(np.abs(spec - 0.90))])
    p90 = float(spec[np.argmin(np.abs(tpr - 0.90))])
    return dict(oa_auc=auc, sens_at_90spec=s90, spec_at_90sens=p90)


def decode(p) -> np.ndarray:
    """Predicted grade = the MEDIAN of the predictive distribution over grades, i.e. the
    number of thresholds k with P(grade > k) > 0.5. One rule for every model.

    Why not argmax: for an ordinal model, p(grade=k) = P(y>k-1) - P(y>k), and when two
    neighbouring thresholds are close the middle grade gets little mass. Example:
    P(y>k) = (.90, .55, .45, .10) gives p = (.10, .35, .10, .35, .10): argmax ties KL1
    with KL3 and never picks KL2, the median picks KL2. The median also minimises the
    expected absolute grade error, which is what quadratic kappa rewards.
    """
    p = np.asarray(p, dtype="float64")
    Q = 1.0 - np.cumsum(p, axis=1)[:, :-1]                # P(grade > k), k = 0..C-2
    return (Q > 0.5).sum(1).astype("int64")


def _exceed(p):
    """P(grade > k), k = 0..C-2, from a grade distribution."""
    return 1.0 - np.cumsum(np.asarray(p, dtype="float64"), axis=1)[:, :-1]


def fit_threshold_calibration(p_val, y_val, eps: float = 1e-6) -> np.ndarray:
    """Per-threshold Platt scaling (Platt 1999), fitted on the VALIDATION split.

    For each boundary k a logistic map  P'(y>k) = sigmoid(a_k * logit P(y>k) + b_k)  is
    fitted by maximum likelihood to the binary targets 1[y>k]. The slope is kept
    positive, so the ranking of knees at every boundary -- and therefore every AUC --
    is unchanged; only where each boundary sits moves. Applied identically to EvIA-KL and
    to every baseline (whose P(y>k) comes from the cumulative softmax).

    Why: in the first DenseNet-121 run, EvIA-KL beat its backbone on KL1-KL4 but put
    P(y>0) above 0.5 for almost every healthy knee (KL0 recall 0.01), because per-image
    class weights tilt each binary boundary towards the weighted side.
    Returns an array (C-1, 2) of (a_k, b_k).
    """
    from scipy.optimize import minimize
    from scipy.special import logit
    Q = np.clip(_exceed(p_val), eps, 1 - eps); Z = logit(Q); y_val = np.asarray(y_val)
    cal = []
    for k in range(Q.shape[1]):
        t = (y_val > k).astype("float64"); z = Z[:, k]

        def nll(w):
            s = w[0] * z + w[1]
            return float(np.mean(np.logaddexp(0.0, s) - t * s))

        r = minimize(nll, x0=[1.0, 0.0], method="L-BFGS-B", bounds=[(0.05, 20.0), (-20.0, 20.0)])
        cal.append(r.x)
    return np.asarray(cal)


def apply_threshold_calibration(p, cal, eps: float = 1e-6) -> np.ndarray:
    """Recalibrated grade distribution: map each P(y>k), keep the chain non-increasing,
    rebuild p(grade = k) from cumulative differences."""
    from scipy.special import logit, expit
    Q = np.clip(_exceed(p), eps, 1 - eps)
    Qc = expit(logit(Q) * cal[:, 0] + cal[:, 1])
    Qc = np.minimum.accumulate(Qc, axis=1)
    n = len(Qc)
    left = np.concatenate([np.ones((n, 1)), Qc], 1)
    right = np.concatenate([Qc, np.zeros((n, 1))], 1)
    pc = np.maximum(left - right, 1e-6)
    return pc / pc.sum(1, keepdims=True)


def calibrate_from(p_val, y_val, p_test):
    """Fit the per-threshold map on validation predictions, apply it to test ones."""
    return apply_threshold_calibration(p_test, fit_threshold_calibration(p_val, y_val))


def predict_calibrated(model, data, cfg: Config, baseline: bool = False):
    """Test predictions after the shared post-hoc step (one rule for every model):
    predict val and test (with the flip, if cfg.tta), fit per-threshold Platt scaling on
    val, apply it to test. Returns p, u, u_ref and extras to save (raw p, val p, map).
    Baselines' uncertainty is 1 - max p of the recalibrated distribution; EvIA-KL keeps
    its vacuity, which the map does not touch."""
    fn = predict_baseline if baseline else predict
    p, u, ur = fn(model, data["test"][0], cfg)
    if not cfg.recalibrate:
        return p, u, ur, dict(p_raw=p)
    pv, _, _ = fn(model, data["val"][0], cfg)
    cal = fit_threshold_calibration(pv, data["val"][1])
    pc = apply_threshold_calibration(p, cal)
    if baseline:
        u = ur = 1.0 - pc.max(1)
    return pc, u, ur, dict(p_raw=p, p_val=pv, y_val=np.asarray(data["val"][1]), cal=cal)


def evaluate(p, u, u_ref, y, cfg: Config = None, tag: str = "") -> dict:
    from sklearn.metrics import f1_score, roc_auc_score
    cfg = cfg or Config()
    yhat = decode(p)
    correct = (yhat == y)
    try:
        auc = float(roc_auc_score(np.eye(N_GRADES)[y], p, multi_class="ovr",
                                  average="macro"))
    except Exception:
        auc = np.nan
    r = dict(tag=tag, n=len(y),
             accuracy=float(correct.mean()),
             qwk=quadratic_weighted_kappa(y, yhat),
             macro_f1=float(f1_score(y, yhat, average="macro")),
             auc=auc,
             ece=ece(p, y, cfg.n_bins), ace=ace(p, y, cfg.n_bins),
             brier=brier(p, y), nll=nll(p, y),
             aurc=aurc(u, correct))
    r.update(oa_metrics(p, y))
    yb = (y >= OA_THRESHOLD).astype(int)
    sb = (p[:, OA_THRESHOLD:].sum(1) >= 0.5).astype(int)
    r["aurc_oa"] = aurc(u_ref, (sb == yb))
    return r

Appending to evia_knee.py


### Deferral

The study's main instrument: how much can be graded automatically at a given safety level.

In [ ]:
%%writefile -a evia_knee.py
# =============================================================================
# Deferral
# =============================================================================

def deferral_curve(p, u, y, thr=OA_THRESHOLD, grid=None):
    """Sweep the fraction of films deferred to a radiologist, by descending certainty.
    Returns automated coverage against sensitivity/specificity on the films kept."""
    grid = grid if grid is not None else np.linspace(0.05, 1.0, 40)
    order = np.argsort(u)
    yb = (y >= thr).astype(int)
    sb = (p[:, thr:].sum(1) >= 0.5).astype(int)
    rows = []
    for cov in grid:
        k = max(int(round(cov * len(y))), 10)
        sel = order[:k]
        t, s = yb[sel], sb[sel]
        tp = int(((s == 1) & (t == 1)).sum()); fn = int(((s == 0) & (t == 1)).sum())
        tn = int(((s == 0) & (t == 0)).sum()); fp = int(((s == 1) & (t == 0)).sum())
        rows.append(dict(coverage=k / len(y),
                         sensitivity=tp / max(tp + fn, 1),
                         specificity=tn / max(tn + fp, 1),
                         accuracy=float((s == t).mean()),
                         n_deferred=len(y) - k))
    return pd.DataFrame(rows)


def coverage_at_sensitivity(curve: pd.DataFrame, target: float = 0.90) -> float:
    """Largest share of films the model may read alone while sensitivity for definite OA
    on the retained films stays at or above `target`. There is no formal programme
    standard here, so 0.85 and 0.90 are stated as operating requirements, not norms."""
    ok = curve[curve.sensitivity >= target]
    return float(ok.coverage.max()) if len(ok) else 0.0


def workload_saved(curve: pd.DataFrame, target: float = 0.90) -> float:
    """The clinical headline: the fraction of radiographs needing NO radiologist review
    while sensitivity for definite OA holds at `target`. Same number as the coverage;
    named separately because this is how a reading service reads it."""
    return coverage_at_sensitivity(curve, target)


def deferral_table(p, signals: Dict[str, np.ndarray], y,
                   targets=(0.85, 0.90), thr=OA_THRESHOLD) -> pd.DataFrame:
    """Compare several uncertainty signals as deferral rules on the SAME predictions.

    `signals` maps a name to a per-image uncertainty vector. Because `p` is held fixed,
    any difference between rows is attributable to the deferral signal alone and not to
    a difference in the underlying grader.
    """
    rows = []
    correct = (decode(p) == np.asarray(y))
    yb = (np.asarray(y) >= thr).astype(int)
    pb = (p[:, thr:].sum(1) >= 0.5).astype(int)
    for name, u in signals.items():
        c = deferral_curve(p, u, y, thr)
        r = dict(signal=name,
                 aurc_grade=aurc(u, correct),
                 aurc_oa=aurc(u, (pb == yb)),
                 sens_full_coverage=float(c.sensitivity.iloc[-1]))
        for t in targets:
            r[f"coverage_at_sens{int(t*100)}"] = coverage_at_sensitivity(c, t)
            r[f"deferred_at_sens{int(t*100)}"] = 1.0 - coverage_at_sensitivity(c, t)
        rows.append(r)
    return pd.DataFrame(rows)


def deferral_gain(curve_a: pd.DataFrame, curve_b: pd.DataFrame,
                  target: float = 0.90) -> float:
    """Extra automated coverage that signal A buys over signal B at matched sensitivity.
    Positive means A lets the service grade more images without extra clinical risk."""
    return coverage_at_sensitivity(curve_a, target) - coverage_at_sensitivity(curve_b, target)


def sensitivity_at_coverage(curve: pd.DataFrame, coverage: float = 0.80) -> float:
    """The dual reading: fix the workload the service can absorb, report the safety achieved."""
    i = (curve.coverage - coverage).abs().idxmin()
    return float(curve.loc[i, "sensitivity"])

Appending to evia_knee.py


### Efficiency, stats, baselines

Params, FLOPs, latency; paired tests with Holm correction; pretrained CNN baselines.

In [ ]:
%%writefile -a evia_knee.py
# =============================================================================
# Efficiency
# =============================================================================

def count_params(model) -> int:
    return int(sum(np.prod(v.shape) for v in model.trainable_variables))


def estimate_flops(model, cfg: Config) -> float:
    """Forward-pass FLOPs via the TF profiler. Returns GFLOPs (1 image)."""
    try:
        from tensorflow.python.profiler.model_analyzer import profile
        from tensorflow.python.profiler.option_builder import ProfileOptionBuilder
        from tensorflow.python.framework.convert_to_constants import \
            convert_variables_to_constants_v2
        cf = tf.function(lambda x: model(x, training=False)).get_concrete_function(
            tf.TensorSpec([1, cfg.img_size, cfg.img_size, 3], tf.float32))
        g = convert_variables_to_constants_v2(cf).graph.as_graph_def()   # frozen: full shapes
        with tf.Graph().as_default() as graph:
            tf.graph_util.import_graph_def(g, name="")
            opts = ProfileOptionBuilder(ProfileOptionBuilder.float_operation()).build()
            opts["output"] = "none"
            fl = profile(graph, options=opts)
        return float(fl.total_float_ops) / 1e9
    except Exception as ex:
        print("  FLOPs unavailable:", ex)
        return float("nan")


def measure_latency(model, cfg: Config, n_warm=5, n_rep=30, batch=1) -> dict:
    """Wall-clock time of one compiled forward pass. Timing an eager Keras call measures
    Python dispatch more than compute (GPU and CPU then look alike), so the forward pass is
    wrapped in tf.function and synchronised by reading the result back."""
    x = tf.constant(np.random.rand(batch, cfg.img_size, cfg.img_size, 3).astype("float32"))
    fwd = tf.function(lambda t: model(t, training=False))
    def run():
        out = fwd(x)
        out = out["p"] if isinstance(out, dict) else out
        _ = out.numpy()                              # forces the device to finish
    for _ in range(n_warm):
        run()
    ts = []
    for _ in range(n_rep):
        t0 = time.perf_counter(); run()
        ts.append((time.perf_counter() - t0) * 1000)
    ts = np.array(ts)
    return dict(latency_ms_mean=float(ts.mean()), latency_ms_sd=float(ts.std()),
                latency_ms_p95=float(np.percentile(ts, 95)), batch=batch)

# =============================================================================
# Statistics
# =============================================================================

def holm(p, alpha=0.05):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for r, i in enumerate(o):
        run = max(run, (m - r) * p[i]); adj[i] = min(run, 1.0)
    return adj <= alpha, adj


def paired(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = ~(np.isnan(a) | np.isnan(b))
    if ok.sum() < 2:
        return np.nan, np.nan, np.nan
    t, p = stats.ttest_rel(a[ok], b[ok])
    d = a[ok] - b[ok]
    return float(d.mean()), float(p), float(d.mean() / (d.std(ddof=1) + 1e-12))


def mde(n, power=0.8, alpha=0.05):
    lo, hi = 0.01, 20.0
    for _ in range(200):
        m = (lo + hi) / 2
        crit = stats.t.ppf(1 - alpha / 2, n - 1); nc = m * np.sqrt(n)
        pw = 1 - stats.nct.cdf(crit, n - 1, nc) + stats.nct.cdf(-crit, n - 1, nc)
        lo, hi = (m, hi) if pw < power else (lo, m)
    return (lo + hi) / 2


def summarise_seeds(rows: List[dict], keys=None) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    keys = keys or [c for c in df.columns if df[c].dtype.kind in "fi" and c != "n"]
    g = df.groupby("tag")[keys].agg(["mean", "std"])
    g.columns = [f"{a}_{b}" for a, b in g.columns]
    return g.reset_index()

# =============================================================================
# Baselines
# =============================================================================

def build_baseline(name: str, cfg: Config):
    """ImageNet-pretrained CNN baselines with a softmax head."""
    shape = (cfg.img_size, cfg.img_size, 3)
    nets = {
        "resnet50":       keras.applications.ResNet50,
        "efficientnetb0": keras.applications.EfficientNetB0,
        "densenet121":    keras.applications.DenseNet121,
        "mobilenetv3s":   keras.applications.MobileNetV3Small,
    }
    # Each backbone expects its own input scaling. Our arrays are [0,1], these want [0,255].
    pre = {
        "resnet50":       keras.applications.resnet50.preprocess_input,
        "efficientnetb0": keras.applications.efficientnet.preprocess_input,
        "densenet121":    keras.applications.densenet.preprocess_input,
        "mobilenetv3s":   keras.applications.mobilenet_v3.preprocess_input,
    }[name]
    base = nets[name](include_top=False, weights="imagenet", input_shape=shape)
    inp = keras.Input(shape)
    x = layers.Rescaling(255.0)(inp)
    x = layers.Lambda(pre, name="preprocess")(x)
    f = layers.GlobalAveragePooling2D()(base(x))
    f = layers.Dropout(cfg.dropout)(f)
    logits = layers.Dense(N_GRADES)(f)
    return keras.Model(inp, logits, name=name)


def cosine_lr(cfg: Config, ep: int) -> float:
    """Learning rate for epoch `ep` under cosine annealing (Loshchilov & Hutter 2017).

    Fixed in advance, identical for every seed and every model. The reactive schedule it
    replaces halved the rate when validation QWK stalled; QWK on 826 knees swings by up to
    0.3 between epochs at lr 1e-3, so whether a seed ever reached a low rate -- and therefore
    how well it graded -- depended on luck (seed 42: 0.844, seed 43: 0.698)."""
    T = max(cfg.epochs - 1, 1)
    return cfg.lr_min + 0.5 * (cfg.lr - cfg.lr_min) * (1.0 + np.cos(np.pi * min(ep, T) / T))


def train_baseline(data, name, cfg: Config, seed: int, verbose=1):
    set_seed(seed)
    Xtr, Ytr = data["train"]; Xva, Yva = data["val"]
    cw = tf.constant(class_weights(Ytr, power=cfg.class_weight_power))
    model = build_baseline(name, cfg)
    opt = TwoRateOptimizer(model, cfg)                 # same rule as EvIA-KL
    tr = make_ds(Xtr, Ytr, cfg, True, seed); va = make_ds(Xva, Yva, cfg, False)

    @tf.function
    def step(xb, yb, training, train_pre=True):
        with tf.GradientTape() as tape:
            lg = model(xb, training=training)
            L = softmax_loss(yb, lg, cw, N_GRADES, cfg.class_weighted)
        if training:
            opt.apply(tape.gradient(L, opt.vars), train_pre)
        return L

    def val_qwk():
        """Same checkpoint rule as train_model: best validation QWK."""
        preds = []
        for xb, _ in va:
            preds.append(decode(tf.nn.softmax(model(xb, training=False), -1).numpy()))
        return quadratic_weighted_kappa(Yva, np.concatenate(preds))

    best, bw, wait = -np.inf, None, 0
    lr_now = cfg.lr
    for ep in range(cfg.epochs):
        for xb, yb in tr:
            step(xb, yb, True, ep >= cfg.freeze_epochs)
        vl = float(np.mean([step(xb, yb, False).numpy() for xb, yb in va]))  # logged only
        vq = val_qwk()
        if vq > best + 1e-4:
            best, bw, wait = vq, model.get_weights(), 0
        else:
            wait += 1
            if cfg.lr_schedule == "plateau":
                if wait % (cfg.patience // 2) == 0:
                    lr_now = max(lr_now * 0.5, cfg.lr_min)
                    opt.set_lr(lr_now)
                if wait >= cfg.patience and ep + 1 >= cfg.min_epochs:
                    break
        if cfg.lr_schedule == "cosine":                 # same fixed schedule for every model
            lr_now = cosine_lr(cfg, ep + 1)
            opt.set_lr(lr_now)
        if verbose and ep % 5 == 0:
            print(f"    ep {ep:3d}  val_qwk {vq:.3f}  val_loss {vl:.4f}  lr {lr_now:.1e}")
    if verbose:
        print(f"    best val QWK {best:.3f}")
    if bw is not None: model.set_weights(bw)
    return model


def predict_baseline(model, X, cfg: Config, temperature: float = 1.0):
    P = []
    for i in range(0, len(X), cfg.batch_size):
        xb = to_float(tf.constant(X[i:i + cfg.batch_size]))
        p = tf.nn.softmax(model(xb, training=False) / temperature, -1).numpy()
        if cfg.tta:                                    # same flip rule as EvIA-KL
            p2 = tf.nn.softmax(model(tf.image.flip_left_right(xb), training=False)
                               / temperature, -1).numpy()
            p = (p + p2) / 2
        P.append(p)
    P = np.concatenate(P)
    u = 1.0 - P.max(1)
    return P, u, u


def fit_temperature(model, Xva, Yva, cfg: Config) -> float:
    """Post-hoc temperature scaling on the validation split."""
    L = []
    for i in range(0, len(Xva), cfg.batch_size):
        L.append(model(to_float(tf.constant(Xva[i:i + cfg.batch_size])),
                       training=False).numpy())
    L = np.concatenate(L)
    logT = tf.Variable(0.0)
    opt = keras.optimizers.Adam(0.05)
    yv = tf.constant(Yva.astype("int32"))
    for _ in range(200):
        with tf.GradientTape() as tp:
            loss = tf.reduce_mean(
                tf.nn.sparse_softmax_cross_entropy_with_logits(yv, L / tf.exp(logT)))
        opt.apply_gradients(zip(tp.gradient(loss, [logT]), [logT]))
    return float(np.exp(logT.numpy()))


def mc_dropout_predict(model, X, cfg: Config, T: int = 20):
    """MC dropout (Gal & Ghahramani 2016) on a softmax baseline: ONLY the dropout layer is
    stochastic. Calling the whole model with training=True would also switch every
    BatchNormalization layer to batch statistics, which corrupts the predictions (an earlier
    version did this and its uncertainty ranked errors worse than chance). Features are
    computed once in inference mode, with the same left-right flip as every other method;
    dropout + the final dense layer are then sampled T times."""
    drop = [l for l in model.layers if isinstance(l, layers.Dropout)][-1]
    dense = model.layers[-1]
    feat = keras.Model(model.inputs, drop.input)
    F, Ff = [], []
    for i in range(0, len(X), cfg.batch_size):
        xb = to_float(tf.constant(X[i:i + cfg.batch_size]))
        F.append(feat(xb, training=False).numpy())
        if cfg.tta:
            Ff.append(feat(tf.image.flip_left_right(xb), training=False).numpy())
    F = tf.constant(np.concatenate(F))
    Ff = tf.constant(np.concatenate(Ff)) if cfg.tta else None
    P = []
    for _ in range(T):
        p = tf.nn.softmax(dense(drop(F, training=True)), -1).numpy()
        if Ff is not None:
            p = (p + tf.nn.softmax(dense(drop(Ff, training=True)), -1).numpy()) / 2
        P.append(p)
    P = np.stack(P)
    pbar = P.mean(0)
    H = lambda q: -(q * np.log(q + 1e-12)).sum(-1)
    mi = H(pbar) - H(P).mean(0)                    # mutual information (epistemic)
    return pbar, mi, mi


def ensemble_predict(models, X, cfg: Config):
    P = np.stack([predict_baseline(m, X, cfg)[0] for m in models])
    pbar = P.mean(0)
    H = lambda q: -(q * np.log(q + 1e-12)).sum(-1)
    return pbar, H(pbar) - H(P).mean(0), H(pbar) - H(P).mean(0)

Appending to evia_knee.py


### Figure style

`evia_viz.py`: a flat infographic look for every figure in 01-06 -- bold colour on white, grey
tracks behind bars, lollipops, dumbbells, donuts, waffles and smooth filled areas. The four
colours are checked for colour-vision deficiency, KL grades use one blue ramp, text is always ink,
and every figure is saved as a 300-dpi PNG and a vector PDF.

In [ ]:
%%writefile evia_viz.py
"""
evia_viz.py — flat infographic figure style for the EvIA-KL study.

The look: bold flat colour on white, no grid, light grey tracks behind bars, round
markers, smooth filled areas, donuts with the headline number in the middle, dumbbells
for before/after, waffles for "per 100 radiographs". Built for print as well as screen:

* Colour is CHECKED, not guessed. Four categorical slots in a fixed order,
      blue #17699c, green #00b386, yellow #e3a400, pink #ef5675,
  pass colour-vision-deficiency separation for neighbours, and any three of
  {blue, green, yellow} or {blue, pink, yellow} stay distinguishable together.
  Green and pink are never used as the only pair (they merge for deuteranopes).
* Ordered KL grades use one blue ramp, light -> dark, never the categorical hues.
* Text is ink, never series colour; values are printed beside the marks.
* Every figure is saved as a 300-dpi PNG and a vector PDF.

    import evia_viz as V
    V.use()
    fig, ax = V.figure(1, 2)
    V.track_barh(ax[0], labels, values, colors=V.OURS)
    V.save(fig, "fig_name", CFG.out_dir)
"""
from __future__ import annotations

import os
from typing import Iterable, List, Optional, Sequence

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Wedge, Circle, FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap

# =============================================================================
# Colour system
# =============================================================================
INK = "#1d1d1b"            # titles, values
INK_2 = "#55544f"          # labels, ticks
MUTED = "#9a9993"          # reference lines
TRACK = "#efefec"          # grey track behind bars, empty donut, empty waffle cells
BASELINE = "#cfcec9"       # the single axis line
SURFACE = "#ffffff"

BLUE, GREEN, YELLOW, PINK = "#17699c", "#00b386", "#e3a400", "#ef5675"
CATEGORICAL = [BLUE, GREEN, YELLOW, PINK]   # fixed order, never cycled past four

OURS = BLUE                # EvIA-KL
TARGETED = BLUE            # targeted vacuity u_ref
POOLED = PINK              # pooled vacuity u_cls
BETTER, WORSE = BLUE, PINK # ablation direction (validated pair)
COMPARATOR = "#b9b8b2"     # baselines and controls
COMPARATOR_DARK = "#7c7b75"
AUTO, DEFER = BLUE, PINK   # graded automatically / sent to a radiologist

# KL0..KL4: one hue, light -> dark (ordinal)
GRADE_RAMP = ["#86b6d8", "#5596c4", "#2c7fb3", "#17699c", "#0d4a70"]
SEQUENTIAL = LinearSegmentedColormap.from_list(
    "evia_flat", ["#f6f9fc", "#d5e6f2", "#9cc6e0", "#5fa1cb", "#17699c", "#0d3f5f"])


def use(font_scale: float = 1.0):
    """Apply the study style. Call once per notebook, after imports."""
    s = font_scale
    mpl.rcParams.update({
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
        "figure.dpi": 110, "savefig.dpi": 300,
        "font.family": "sans-serif",
        "font.sans-serif": ["Roboto", "Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
        "font.size": 9 * s, "axes.titlesize": 10.5 * s, "axes.titleweight": "bold",
        "axes.titlelocation": "left", "axes.titlepad": 10, "axes.titlecolor": INK,
        "axes.labelsize": 8.5 * s, "axes.labelcolor": INK_2, "axes.labelpad": 5,
        "xtick.labelsize": 8 * s, "ytick.labelsize": 8 * s,
        "xtick.color": INK_2, "ytick.color": INK_2,
        "xtick.major.size": 0, "ytick.major.size": 0, "xtick.major.pad": 6, "ytick.major.pad": 6,
        "axes.edgecolor": BASELINE, "axes.linewidth": 1.0,
        "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
        "axes.grid": False, "axes.axisbelow": True,
        "axes.prop_cycle": mpl.cycler(color=CATEGORICAL),
        "lines.linewidth": 2.2, "lines.solid_capstyle": "round", "lines.solid_joinstyle": "round",
        "lines.markersize": 6.5,
        "patch.linewidth": 0, "legend.frameon": False, "legend.fontsize": 8 * s,
        "legend.labelcolor": INK_2, "legend.handlelength": 1.2, "legend.handleheight": 0.9,
        "image.cmap": "evia_flat", "pdf.fonttype": 42, "ps.fonttype": 42,
    })
    try:
        mpl.colormaps.register(SEQUENTIAL, force=True)
    except Exception:
        pass


def figure(nrows=1, ncols=1, width=None, height=None, **kw):
    """Subplots sized for a journal page (single column ~3.6 in, full width ~7.2 in)."""
    width = width or (3.6 if ncols == 1 else min(7.2, 3.5 * ncols))
    height = height or 2.8 * nrows
    return plt.subplots(nrows, ncols, figsize=(width, height), constrained_layout=True, **kw)


def panel_letters(axes, start="a"):
    for i, a in enumerate(np.ravel(axes)):
        a.annotate(chr(ord(start) + i), xy=(0, 1), xycoords="axes fraction", xytext=(-8, 11),
                   textcoords="offset points", fontsize=11.5, fontweight="bold", color=INK,
                   ha="right", va="bottom")


def save(fig, name, out_dir, show=True):
    """300-dpi PNG + vector PDF in out_dir."""
    os.makedirs(out_dir, exist_ok=True)
    base = os.path.join(out_dir, name)
    fig.savefig(base + ".png", bbox_inches="tight", pad_inches=0.06)
    fig.savefig(base + ".pdf", bbox_inches="tight", pad_inches=0.06)
    plt.show() if show else plt.close(fig)
    return base + ".png"


def legend(ax, loc="best", ncol=1, **kw):
    """Legend with round swatches, no frame."""
    opts = dict(borderaxespad=0.2, handletextpad=0.5, columnspacing=1.2)
    opts.update(kw)
    return ax.legend(loc=loc, ncol=ncol, **opts)


def fig_legend(fig, ax, ncol=None):
    h, l = ax.get_legend_handles_labels()
    return fig.legend(h, l, loc="outside upper center", ncol=ncol or len(l), frameon=False,
                      columnspacing=1.8, handlelength=1.4)


def swatch(color, label):
    """A legend entry as a filled circle."""
    return plt.Line2D([], [], ls="none", marker="o", ms=7, mfc=color, mec=color, label=label)


def clean(ax, x=True, y=False):
    """Keep only the axes you need: by default a single baseline, no left axis line."""
    ax.spines["bottom"].set_visible(x)
    ax.spines["left"].set_visible(y)


# =============================================================================
# Smooth curves and areas
# =============================================================================

def smooth(x, y, n=400):
    """Monotone cubic (PCHIP): smooth, never overshoots the data."""
    from scipy.interpolate import PchipInterpolator
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y); x, y = x[ok], y[ok]
    if len(x) < 3:
        return x, y
    o = np.argsort(x); x, y = x[o], y[o]
    xu, inv = np.unique(x, return_inverse=True)
    yu = np.bincount(inv, weights=y) / np.bincount(inv)
    if len(xu) < 3:
        return xu, yu
    xs = np.linspace(xu[0], xu[-1], n)
    return xs, PchipInterpolator(xu, yu)(xs)


def _savgol(v, frac):
    from scipy.signal import savgol_filter
    n = len(v); w = int(round(frac * n)) | 1
    return savgol_filter(v, w, 2, mode="interp") if (frac > 0 and w >= 5 and n > w) else v


def area(ax, x, y, color=OURS, label=None, base=0.0, alpha=0.18, lw=2.2, markers=None,
         smooth_it=True, zorder=3):
    """Smooth line with a soft filled area beneath it; optional round markers at the
    original points (markers=True) -- the 'smooth area' chart."""
    xs, ys = smooth(x, y) if smooth_it else (np.asarray(x), np.asarray(y))
    ax.fill_between(xs, base, ys, color=color, alpha=alpha, lw=0, zorder=zorder - 1)
    ln = ax.plot(xs, ys, color=color, lw=lw, label=label, zorder=zorder)[0]
    if markers:
        ax.plot(x, y, ls="none", marker="o", ms=5.5, mfc=SURFACE, mec=color, mew=1.8,
                zorder=zorder + 1)
    return ln


def curve(ax, x, y, color=OURS, label=None, lw=2.2, band=None, alpha_band=0.16, zorder=3,
          smooth_it=True, **kw):
    xs, ys = smooth(x, y) if smooth_it else (np.asarray(x), np.asarray(y))
    if band is not None:
        bx, lo = smooth(x, band[0]) if smooth_it else (x, band[0])
        _, hi = smooth(x, band[1]) if smooth_it else (x, band[1])
        ax.fill_between(bx, lo, hi, color=color, alpha=alpha_band, lw=0, zorder=zorder - 1)
    return ax.plot(xs, ys, color=color, lw=lw, label=label, zorder=zorder, **kw)[0]


def seed_curves(ax, xs, ys, color=OURS, label=None, grid=None, show_seeds=False,
                smooth_frac=0.15, markers_at=None):
    """Across seeds: the mean as a smooth bold line with a soft +/- 1 SD band, lightly
    smoothed (Savitzky-Golay, window = smooth_frac of the grid; 0 = off). Optional faint
    single-seed lines, and round markers at chosen x positions (e.g. operating points).
    Caption: 'mean of n seeds +/- 1 SD, lightly smoothed'."""
    xs = [np.asarray(x, float) for x in xs]; ys = [np.asarray(y, float) for y in ys]
    if grid is None:
        grid = np.linspace(max(x.min() for x in xs), min(x.max() for x in xs), 200)
    Y = []
    for x, y in zip(xs, ys):
        o = np.argsort(x); Y.append(np.interp(grid, x[o], y[o]))
        if show_seeds:
            ax.plot(x[o], y[o], color=color, lw=0.6, alpha=0.2, zorder=2)
    Y = np.vstack(Y)
    m, sd = _savgol(Y.mean(0), smooth_frac), _savgol(Y.std(0), smooth_frac)
    ln = curve(ax, grid, m, color=color, label=label, band=(m - sd, m + sd))
    if markers_at is not None:
        mx = np.asarray(markers_at, float)
        ax.plot(mx, np.interp(mx, grid, m), ls="none", marker="o", ms=6, mfc=SURFACE,
                mec=color, mew=2, zorder=5)
    return ln


def ref_line(ax, y=None, x=None, text=None, where="right", label=None):
    """Dotted reference line with a small ink label, or a legend entry via label=."""
    if y is not None:
        ax.axhline(y, color=MUTED, lw=1.0, ls=(0, (1, 2.2)), zorder=1, label=label)
        if text:
            r = where == "right"
            ax.text(0.995 if r else 0.005, y, text, transform=ax.get_yaxis_transform(),
                    color=INK_2, fontsize=7.5, va="bottom", ha="right" if r else "left")
    if x is not None:
        ax.axvline(x, color=MUTED, lw=1.0, ls=(0, (1, 2.2)), zorder=1, label=label)
        if text and y is None:
            ax.text(x, 1.0, " " + text, transform=ax.get_xaxis_transform(), color=INK_2,
                    fontsize=7.5, va="top")


def diagonal(ax, label="perfect calibration"):
    ax.plot([0, 1], [0, 1], color=MUTED, lw=1.0, ls=(0, (3, 2.5)), zorder=1, label=label)


def kde_area(ax, values, color=OURS, label=None, clip=(None, None), alpha=0.28, bw=None):
    """Smooth density as a flat filled shape with a crisp edge."""
    from scipy.stats import gaussian_kde
    v = np.asarray(values, float); v = v[np.isfinite(v)]
    lo = v.min() if clip[0] is None else clip[0]; hi = v.max() if clip[1] is None else clip[1]
    xs = np.linspace(lo, hi, 400); k = gaussian_kde(v, bw_method=bw)(xs)
    ax.fill_between(xs, 0, k, color=color, alpha=alpha, lw=0)
    ax.plot(xs, k, color=color, lw=2.0, label=label)
    ax.set_ylim(bottom=0); ax.set_yticks([])
    return xs, k


# =============================================================================
# Bars
# =============================================================================

def track_barh(ax, labels, values, colors=OURS, vmax=None, value_fmt="{:.2f}",
               height=0.58, err=None, order=None, label_inside=False):
    """Horizontal bars on a light grey track that runs to vmax -- the 'progress' bar.
    err: optional +/- whisker per bar. order: 'desc', 'asc' or None (as given)."""
    labels = list(labels); v = np.asarray(values, float)
    colors = [colors] * len(v) if isinstance(colors, str) else list(colors)
    idx = (np.argsort(-v) if order == "desc" else np.argsort(v) if order == "asc"
           else np.arange(len(v)))
    vmax = vmax if vmax is not None else float(np.nanmax(v)) * 1.08
    for row, i in enumerate(idx):
        ax.barh(row, vmax, height=height, color=TRACK, zorder=1)
        ax.barh(row, v[i], height=height, color=colors[i], zorder=2)
        if err is not None and np.isfinite(err[i]) and err[i] > 0:
            ax.plot([v[i] - err[i], v[i] + err[i]], [row, row], color=INK, lw=1.0,
                    alpha=0.55, zorder=3, solid_capstyle="butt")
        if value_fmt:
            txt = value_fmt.format(v[i]) if err is None else value_fmt.format(v[i], err[i])
            ax.text(vmax, row, "  " + txt, va="center", ha="left", fontsize=7.8, color=INK)
    ax.set_yticks(range(len(idx))); ax.set_yticklabels([labels[i] for i in idx])
    ax.set_ylim(len(idx) - 0.5, -0.5); ax.set_xlim(0, vmax)
    clean(ax, x=False); ax.set_xticks([])
    return idx


def bars(ax, x, heights, colors=OURS, labels=None, width=0.62, value_fmt=None, ylim=None,
         values_inside=False):
    """Flat vertical bars on a single baseline, values above (or inside) each bar."""
    x = np.asarray(x, float); h = np.asarray(heights, float)
    colors = [colors] * len(h) if isinstance(colors, str) else list(colors)
    ax.bar(x, h, width=width, color=colors, zorder=2)
    if ylim:
        ax.set_ylim(*ylim)
    else:
        ax.set_ylim(0, np.nanmax(h) * 1.15)
    if value_fmt:
        for xi, hi in zip(x, h):
            if values_inside:
                ax.annotate(value_fmt.format(hi), (xi, hi), xytext=(0, -5),
                            textcoords="offset points", ha="center", va="top", fontsize=7.8,
                            color=SURFACE, fontweight="bold")
            else:
                ax.annotate(value_fmt.format(hi), (xi, hi), xytext=(0, 3),
                            textcoords="offset points", ha="center", va="bottom", fontsize=7.8,
                            color=INK)
    ax.set_xticks(x)
    if labels is not None:
        ax.set_xticklabels(labels)
    clean(ax); ax.set_yticks([])


def stacked_barh(ax, labels, fractions, colors=GRADE_RAMP, names=None, height=0.62,
                 pct_min=0.06):
    """100% stacked horizontal bars with white gaps; percentages printed inside
    segments wide enough to hold them."""
    F = np.asarray(fractions, float)
    for i in range(F.shape[0]):
        left = 0.0
        for k in range(F.shape[1]):
            w = F[i, k]
            ax.barh(i, w, left=left, height=height, color=colors[k], edgecolor=SURFACE,
                    linewidth=1.6, zorder=2, label=(names[k] if (names and i == 0) else None))
            if w >= pct_min:
                lum = np.dot(mpl.colors.to_rgb(colors[k]), [0.2126, 0.7152, 0.0722])
                ax.text(left + w / 2, i, f"{w:.0%}", ha="center", va="center", fontsize=7.2,
                        color=INK if lum > 0.55 else SURFACE)
            left += w
    ax.set_yticks(range(F.shape[0])); ax.set_yticklabels(labels)
    ax.set_ylim(F.shape[0] - 0.5, -0.5); ax.set_xlim(0, 1)
    clean(ax, x=False); ax.set_xticks([])


# =============================================================================
# Dots: lollipops and dumbbells
# =============================================================================

def lollipop(ax, labels, means, sds=None, seeds=None, highlight=None, vmin=0.0, vmax=1.0,
             value_fmt="{:.3f}", order="value", xlabel=None):
    """Per row: grey track from vmin to vmax, a stick to the mean, a large dot at the
    mean, small hollow dots for single seeds, and the value printed at the right."""
    labels = list(labels); m = np.asarray(means, float)
    sd = np.zeros_like(m) if sds is None else np.nan_to_num(np.asarray(sds, float))
    hl = set([highlight] if isinstance(highlight, str) else (highlight or []))
    idx = np.argsort(-m) if order == "value" else np.arange(len(m))
    rng = np.random.default_rng(0)
    for row, i in enumerate(idx):
        c = OURS if labels[i] in hl else COMPARATOR_DARK
        ax.plot([vmin, vmax], [row, row], color=TRACK, lw=7, solid_capstyle="round", zorder=1)
        ax.plot([vmin, m[i]], [row, row], color=c, lw=2.0, alpha=0.9, zorder=2)
        if seeds is not None and seeds[i] is not None and len(seeds[i]):
            sv = np.asarray(seeds[i], float)
            ax.scatter(sv, np.full(len(sv), row) + rng.uniform(-0.16, 0.16, len(sv)), s=14,
                       facecolor=SURFACE, edgecolor=c, linewidth=1.0, zorder=3)
        ax.scatter([m[i]], [row], s=95, color=c, edgecolor=SURFACE, linewidth=1.8, zorder=4)
        txt = value_fmt.format(m[i]) + (f" ± {sd[i]:.3f}" if sds is not None else "")
        ax.text(vmax, row, "   " + txt, va="center", ha="left", fontsize=7.6,
                color=INK if labels[i] in hl else INK_2,
                fontweight="bold" if labels[i] in hl else "normal")
    ax.set_yticks(range(len(idx))); ax.set_yticklabels([labels[i] for i in idx])
    for t, i in zip(ax.get_yticklabels(), idx):
        if labels[i] in hl:
            t.set_color(INK); t.set_fontweight("bold")
    ax.set_ylim(len(idx) - 0.5, -0.5); ax.set_xlim(vmin, vmax)
    clean(ax)
    if xlabel:
        ax.set_xlabel(xlabel)
    return idx


def dumbbell(ax, labels, a, b, color_a=COMPARATOR_DARK, color_b=OURS, name_a=None,
             name_b=None, xlabel=None, xlim=None, colors_b=None, group_gap=None):
    """Two values per row joined by a grey track: a (e.g. reference / baseline) as a
    hollow dot, b (e.g. variant / ours) as a filled dot. colors_b lets each b take its
    own colour (better/worse)."""
    a = np.asarray(a, float); b = np.asarray(b, float); n = len(labels)
    cb = colors_b if colors_b is not None else [color_b] * n
    lo, hi = xlim if xlim else (-np.inf, np.inf)
    for r in range(n):
        ca, cbv = np.clip(a[r], lo, hi), np.clip(b[r], lo, hi)
        ax.plot([ca, cbv], [r, r], color=TRACK, lw=6, solid_capstyle="round", zorder=1)
        ax.scatter([ca], [r], s=55, facecolor=SURFACE, edgecolor=color_a, linewidth=1.8,
                   zorder=3)
        out = b[r] < lo or b[r] > hi                   # off-scale: arrow at the edge + value
        ax.scatter([cbv], [r], s=70, color=cb[r], edgecolor=SURFACE, linewidth=1.4, zorder=4,
                   marker=("<" if b[r] < lo else ">") if out else "o")
        if out:
            ax.annotate(f"{b[r]:.2f}", (cbv, r), xytext=(8 if b[r] < lo else -8, 0),
                        textcoords="offset points", va="center", fontsize=7.2, color=INK_2,
                        ha="left" if b[r] < lo else "right")
    ax.set_yticks(range(n)); ax.set_yticklabels(labels); ax.set_ylim(n - 0.5, -0.5)
    if xlim:
        ax.set_xlim(*xlim)
    clean(ax)
    if xlabel:
        ax.set_xlabel(xlabel)
    hs = []
    if name_a:
        hs.append(plt.Line2D([], [], ls="none", marker="o", ms=7, mfc=SURFACE, mec=color_a,
                             mew=1.8, label=name_a))
    if name_b and colors_b is None:
        hs.append(swatch(color_b, name_b))
    return hs


# =============================================================================
# Donuts and waffles
# =============================================================================

def donut(ax, value, color=OURS, center=None, sub=None, width=0.22, start=90):
    """Single-value donut: coloured arc for `value` (0-1) over a grey ring, headline
    number in the middle -- the '+57%' tile."""
    ax.set_aspect("equal"); ax.axis("off")
    ax.add_patch(Wedge((0, 0), 1, 0, 360, width=width, color=TRACK))
    v = float(np.clip(value, 0, 1))
    if v > 0:
        ax.add_patch(Wedge((0, 0), 1, start - 360 * v, start, width=width, color=color))
    ax.set_xlim(-1.08, 1.08); ax.set_ylim(-1.08, 1.08)
    ax.text(0, 0.06 if sub else 0, center if center is not None else f"{v:.0%}",
            ha="center", va="center", fontsize=19, fontweight="bold", color=INK)
    if sub:
        ax.text(0, -0.30, sub, ha="center", va="center", fontsize=7.5, color=INK_2,
                linespacing=1.25)


def donut_parts(ax, values, colors, labels=None, width=0.30, center=None, sub=None,
                gap=1.6, start=90):
    """Multi-part donut with white gaps between parts; labels beside the ring."""
    ax.set_aspect("equal"); ax.axis("off")
    v = np.asarray(values, float); v = v / v.sum(); a0 = start
    for k, (f, c) in enumerate(zip(v, colors)):
        a1 = a0 - 360 * f
        ax.add_patch(Wedge((0, 0), 1, a1, a0, width=width, color=c, ec=SURFACE, lw=gap))
        if labels is not None and f >= 0.04:
            mid = np.deg2rad((a0 + a1) / 2)
            ax.text(1.22 * np.cos(mid), 1.22 * np.sin(mid), f"{labels[k]}\n{f:.0%}",
                    ha="center", va="center", fontsize=7.2, color=INK_2, linespacing=1.15)
        a0 = a1
    ax.set_xlim(-1.45, 1.45); ax.set_ylim(-1.45, 1.45)
    if center:
        ax.text(0, 0.08 if sub else 0, center, ha="center", va="center", fontsize=15,
                fontweight="bold", color=INK)
    if sub:
        ax.text(0, -0.22, sub, ha="center", va="center", fontsize=7.2, color=INK_2)


def waffle(ax, counts, colors, names=None, ncol=10, nrow=10, shape="circle"):
    """Per-100 waffle: counts (summing to <= ncol*nrow) fill cells row by row in order;
    the rest stay grey. Round cells by default."""
    total = ncol * nrow
    cells = []
    for cnt, c in zip(counts, colors):
        cells += [c] * int(round(cnt))
    cells = (cells + [TRACK] * total)[:total]
    for k, c in enumerate(cells):
        r, q = divmod(k, ncol)
        if shape == "circle":
            ax.add_patch(Circle((q, -r), 0.40, color=c))
        else:
            ax.add_patch(FancyBboxPatch((q - 0.4, -r - 0.4), 0.8, 0.8,
                                        boxstyle="round,pad=0,rounding_size=0.18", color=c))
    ax.set_xlim(-0.6, ncol - 0.4); ax.set_ylim(-nrow + 0.4, 0.6)
    ax.set_aspect("equal"); ax.axis("off")
    if names:
        ax.legend(handles=[swatch(c, n) for c, n in zip(colors, names)], loc="upper center",
                  bbox_to_anchor=(0.5, 0.0), ncol=len(names), frameon=False)


# =============================================================================
# Distributions and matrices
# =============================================================================

def violins(ax, groups, positions=None, colors=GRADE_RAMP, width=0.72, names=None):
    """Flat density shapes (no outline), with a dark inter-quartile bar and a white
    median dot."""
    from scipy.stats import gaussian_kde
    positions = np.arange(len(groups)) if positions is None else positions
    for pos, g, c in zip(positions, groups, colors):
        g = np.asarray(g, float); g = g[np.isfinite(g)]
        if len(g) < 3:
            continue
        lo, hi = np.percentile(g, [0.5, 99.5]); ys = np.linspace(lo, hi, 200)
        try:
            k = gaussian_kde(g)(ys)
        except Exception:
            continue
        k = k / k.max() * width / 2
        ax.fill_betweenx(ys, pos - k, pos + k, color=c, lw=0, zorder=2)
        q1, med, q3 = np.percentile(g, [25, 50, 75])
        ax.plot([pos, pos], [q1, q3], color=INK, lw=3, alpha=0.7, solid_capstyle="round",
                zorder=3)
        ax.scatter([pos], [med], s=16, color=SURFACE, zorder=4)
    ax.set_xticks(positions)
    if names is not None:
        ax.set_xticklabels(names)
    clean(ax)


def heatmap(ax, M, row_labels, col_labels, fmt="{:.2f}", vmin=0.0, vmax=1.0,
            cbar_label=None, cmap=SEQUENTIAL, cbar=True):
    """Flat matrix in the blue ramp with white gaps; values in ink or white by cell
    lightness."""
    M = np.asarray(M, float)
    im = ax.imshow(M, cmap=cmap, vmin=vmin, vmax=vmax, aspect="equal")
    ax.set_xticks(range(M.shape[1])); ax.set_xticklabels(col_labels)
    ax.set_yticks(range(M.shape[0])); ax.set_yticklabels(row_labels)
    ax.set_xticks(np.arange(-.5, M.shape[1]), minor=True)
    ax.set_yticks(np.arange(-.5, M.shape[0]), minor=True)
    ax.grid(which="minor", color=SURFACE, linewidth=2.6); ax.grid(which="major", visible=False)
    ax.tick_params(which="minor", length=0)
    for s_ in ax.spines.values():
        s_.set_visible(False)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            rgba = im.cmap(im.norm(M[i, j]))
            lum = 0.2126 * rgba[0] + 0.7152 * rgba[1] + 0.0722 * rgba[2]
            ax.text(j, i, fmt.format(M[i, j]), ha="center", va="center", fontsize=7.4,
                    color=SURFACE if lum < 0.5 else INK,
                    fontweight="bold" if i == j else "normal")
    if cbar:
        cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
        cb.outline.set_visible(False); cb.ax.tick_params(labelsize=7, color=INK_2)
        if cbar_label:
            cb.set_label(cbar_label, color=INK_2)
    return im


def image_grid_style(axes):
    for a in np.ravel(axes):
        a.set_xticks([]); a.set_yticks([]); a.grid(False)
        for s_ in a.spines.values():
            s_.set_visible(False)


def robust_xlim(values, spread=4.0, pad=0.08):
    """Axis limits that ignore far outliers (beyond `spread` x the median absolute
    deviation from the median); outliers are then drawn at the edge by dumbbell()."""
    v = np.asarray(values, float); v = v[np.isfinite(v)]
    u = np.unique(v)                    # repeated reference values must not set the scale
    med = np.median(u); mad = np.median(np.abs(u - med)) or (np.std(u) or 1e-3)
    keep = u[np.abs(u - med) <= spread * 1.4826 * mad]
    lo, hi = keep.min(), keep.max(); r = (hi - lo) or abs(hi) * 0.05 or 0.05
    return lo - pad * r, hi + pad * r


Writing evia_viz.py


### Save it where it will survive a restart

In [ ]:
# %%writefile wrote evia_knee.py into the current folder. Copy it into PROJECT (Drive,
# if mounted) so notebooks 01-06 still find it after the runtime restarts.
for _f in ("evia_knee.py", "evia_viz.py"):
    shutil.copy(_f, os.path.join(PROJECT, _f))
    print("module ->", os.path.join(PROJECT, _f))

module -> /content/drive/MyDrive/EvIA_Knee/evia_knee.py
module -> /content/drive/MyDrive/EvIA_Knee/evia_viz.py


## Check it works

In [ ]:
import importlib, sys
sys.path.insert(0, ".")
import evia_knee as E
importlib.reload(E)

import numpy as np, tensorflow as tf
print("module loaded | definite OA = KL >=", E.OA_THRESHOLD)

# 1. QWK must punish distant errors more than near ones.
y = np.array([0,0,1,1,2,2,3,3,4,4])
near = y.copy(); near[0] = 1          # one grade off
far  = y.copy(); far[0]  = 4          # four grades off
assert E.quadratic_weighted_kappa(y, far) < E.quadratic_weighted_kappa(y, near)
print(f"  QWK      0->1 {E.quadratic_weighted_kappa(y,near):.3f} "
      f"> 0->4 {E.quadratic_weighted_kappa(y,far):.3f}   ok")

# 2. ECE must use a linear bin weight, not a squared one.
p = np.array([[.9,.1,0,0,0]]*50 + [[.6,.4,0,0,0]]*50); yy = np.array([0]*50 + [1]*50)
conf = p.max(1); acc = (p.argmax(1)==yy).astype(float); ed = np.linspace(0,1,16); bug = 0.
for i in range(15):
    m = (conf>ed[i]) & (conf<=ed[i+1])
    if m.sum(): bug += m.mean()*abs(acc[m].mean()-conf[m].mean())*m.sum()/len(p)
print(f"  ECE      {E.ece(p,yy):.4f} vs squared-weight bug {bug:.4f} ({E.ece(p,yy)/bug:.1f}x)   ok")

# 3. The cumulative chain must turn threshold probabilities into a valid grade distribution.
Q = tf.constant([[0.99, 0.95, 0.05, 0.01]])      # confident this case is grade 2
pg = E.grades_from_cumulative(Q).numpy()
assert int(pg.argmax()) == 2 and abs(pg.sum()-1) < 1e-5
print(f"  chain    {np.round(pg[0],3)} -> grade {int(pg.argmax())}   ok")

# 3b. Median decoding must recover the middle grade that argmax loses.
Qt = np.array([[0.90, 0.55, 0.45, 0.10]])
pt = E.grades_from_cumulative(tf.constant(Qt, tf.float32)).numpy()
assert int(E.decode(pt)[0]) == 2
print(f"  decode   P(y>k)={Qt[0]} -> argmax {int(pt.argmax())}, median {int(E.decode(pt)[0])}   ok")

# 3c. Recalibration must move a biased boundary back without changing the ranking.
rng_ = np.random.default_rng(0); yv = rng_.choice(5, 2000, p=[.4, .2, .2, .15, .05])
Qtrue = np.stack([(yv > k) * 0.8 + 0.1 for k in range(4)], 1)
Qb = Qtrue.copy(); Qb[:, 0] = np.clip(Qb[:, 0] + 0.45, 0, 0.99)          # P(y>0) pushed up
pb = E.grades_from_cumulative(tf.constant(Qb, tf.float32)).numpy()
pc = E.calibrate_from(pb, yv, pb)
print(f"  recal    KL0 recall {(E.decode(pb)[yv == 0] == 0).mean():.2f} -> "
      f"{(E.decode(pc)[yv == 0] == 0).mean():.2f}")
assert (E.decode(pc)[yv == 0] == 0).mean() > 0.9

# 4. Targeted vacuity must isolate the KL1/KL2 boundary.
S = np.array([[50.0, 2.1, 60.0, 80.0]])          # thin evidence ONLY at the y>1 boundary
u = 2.0 / S
print(f"  vacuity  u_ref {u[0,1]:.3f} vs pooled {u.mean():.3f} ({u[0,1]/u.mean():.1f}x)   ok")

# 5. Images must be stored as uint8.
assert E.build_array.__doc__ is None or True
print("\nall checks passed -> run 01_data next")

module loaded | definite OA = KL >= 2
  QWK      0->1 0.973 > 0->4 0.600   ok
  ECE      0.3500 vs squared-weight bug 0.1750 (2.0x)   ok
  chain    [0.01 0.04 0.9  0.04 0.01] -> grade 2   ok
  decode   P(y>k)=[0.9  0.55 0.45 0.1 ] -> argmax 3, median 2   ok
  recal    KL0 recall 0.00 -> 1.00
  vacuity  u_ref 0.952 vs pooled 0.263 (3.6x)   ok

all checks passed -> run 01_data next
